# 01. Dataset Analysis

AI-Hub **'숫자가 포함된 패턴 발화'** 데이터셋의 구조·분포·토큰 길이를 분석하고,  
CoT 학습을 위한 계층적 샘플링(11,900건) 결과를 검증합니다.

In [1]:
import pandas as pd

train_df = pd.read_csv('../data/train_set.csv')
val_df   = pd.read_csv('../data/validation_set.csv')
test_df  = pd.read_csv('../data/test_set.csv')

total = len(train_df) + len(val_df) + len(test_df)
print('=== Dataset Split Statistics ===')
print(f'Total samples : {total:,}')
print(f'  Train       : {len(train_df):,}  ({len(train_df)/total*100:.1f}%)')
print(f'  Validation  :  {len(val_df):,}  ({len(val_df)/total*100:.1f}%)')
print(f'  Test        :  {len(test_df):,}  ({len(test_df)/total*100:.1f}%)')
print(f'\nNumber of pattern categories : {train_df["pattern"].nunique()}')

=== Dataset Split Statistics ===
Total samples : 149,898
  Train       : 119,918  (80.0%)
  Validation  :  14,990  (10.0%)
  Test        :  14,990  (10.0%)

Number of pattern categories : 84


In [2]:
top10 = train_df['pattern'].value_counts().head(10)
print('=== Top-10 Pattern Categories (Train) ===')
for cat, cnt in top10.items():
    print(f'{cat:<20} {cnt:>6,}  ({cnt/len(train_df)*100:.1f}%)')

=== Top-10 Pattern Categories (Train) ===
기타           23,841  (19.9%)
통계-소수-통계     18,120  (15.1%)
날짜-기간-분기      9,853   (8.2%)
금융-금액-금액      8,961   (7.5%)
날짜-날짜-날짜      7,432   (6.2%)
고유어-시간-기수     6,218   (5.2%)
통화-금액-금액      5,774   (4.8%)
날짜-시간-시간      4,903   (4.1%)
명수-인원-인원      4,251   (3.5%)
전화번호-전화번호     3,874   (3.2%)


In [3]:
distilled_df = pd.read_csv('../data/golden_11900_stratified_samples.csv')
print('=== Stratified Distilled Subset (11,900 samples) ===')
print(f'Loaded from: ../data/golden_11900_stratified_samples.csv')
print(f'Total        : {len(distilled_df):,}')
print(f'Categories   : {distilled_df["pattern"].nunique():>6}  (all 84 categories covered)')
cat_counts = distilled_df['pattern'].value_counts()
print(f'Min per cat  : {cat_counts.min():>6}')
print(f'Max per cat  : {cat_counts.max():>6}')
print(f'Mean per cat : {cat_counts.mean():>8.1f}')
print('\nTop-5 in distilled subset:')
for cat, cnt in cat_counts.head(5).items():
    print(f'{cat:<20} {cnt}')

=== Stratified Distilled Subset (11,900 samples) ===
Loaded from: ../data/golden_11900_stratified_samples.csv
Total        : 11,900
Categories   :     84  (all 84 categories covered)
Min per cat  :      1
Max per cat  :    872
Mean per cat :    141.7

Top-5 in distilled subset:
기타           872
통계-소수-통계     661
날짜-기간-분기     358
금융-금액-금액     326
날짜-날짜-날짜     271


In [4]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B', trust_remote_code=True)

def token_stats(df, col='scriptTN'):
    lengths = df[col].dropna().apply(lambda x: len(tokenizer.encode(str(x))))
    return lengths.min(), int(np.percentile(lengths, 50)), int(np.percentile(lengths, 95)), int(np.percentile(lengths, 99)), lengths.max()

print('=== Token Length Analysis (Qwen3 Tokenizer) ===')
print(f'{"Split":<10}| min  | p50  | p95  | p99  | max')
print('-'*10 + '|------|------|------|------|---------')
for name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    mn, p50, p95, p99, mx = token_stats(df)
    print(f'{name:<10}| {mn:>4} | {p50:>4} | {p95:>4} | {p99:>4} | {mx:>6}')

print(f'\n99% of all samples are under 62 tokens.')
print(f'→ Max sequence length 1,408 (Few-Shot) / 2,048 (Few-Shot-CoT) covers 100% without truncation.')

=== Token Length Analysis (Qwen3 Tokenizer) ===
Split     | min  | p50  | p95  | p99  | max
----------|------|------|------|------|---------
Train     |    3 |   19 |   48 |   62 |   326
Validation|    3 |   19 |   48 |   62 |   298
Test      |    3 |   19 |   48 |   62 |   312

99% of all samples are under 62 tokens.
→ Max sequence length 1,408 (Few-Shot) / 2,048 (Few-Shot-CoT) covers 100% without truncation.
